# 컬럼 dtype 결정

DuckDB 기본 추론은 NULL이 있는 정수 컬럼을 전부 DOUBLE 로 잡는다.
실제 값 범위를 보고 최소 타입을 정한다.

In [1]:
import duckdb
from collections import Counter, defaultdict

BASE = "../../data/ieee-fraud-detection"

con = duckdb.connect(":memory:")
con.execute("SET memory_limit='3GB'")

# train, test 데이터셋 확인
con.execute(f"""
    CREATE VIEW t AS
    SELECT * FROM read_csv('{BASE}/train_transaction.csv', sample_size=-1)
    UNION ALL BY NAME
    SELECT * FROM read_csv('{BASE}/test_transaction.csv', sample_size=-1)
""")

In [2]:
# 자동 추론 결과
inferred = {name: typ for name, typ, *_ in con.execute("DESCRIBE t").fetchall()}
Counter(inferred.values())

Counter({'DOUBLE': 376, 'BOOLEAN': 8, 'VARCHAR': 6, 'BIGINT': 4})

In [3]:
# 숫자 컬럼만 확인. VARCHAR / BOOLEAN은 제외
numeric = [c for c, t in inferred.items() if t in ("DOUBLE", "BIGINT", "INTEGER")]
len(numeric)

380

In [4]:
# 컬럼마다 min / max /정수여부 확인
parts = []
for c in numeric:
    parts.append(
        f'MIN("{c}"), MAX("{c}"), BOOL_AND("{c}" = FLOOR("{c}"))'
    )
row = con.execute("SELECT " + ", ".join(parts) + " FROM t").fetchone()

stats = {}
for i, c in enumerate(numeric):
    stats[c] = {"min": row[i * 3], "max": row[i * 3 + 1], "is_int": row[i * 3 + 2]}

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
def pick_type(s: dict) -> str:
    """값 범위로 최소 타입 선정"""
    lo, hi, is_int = s["min"], s["max"], s["is_int"]
    if lo is None:
        return "ALL_NULL"
    if not is_int:
        return "DOUBLE"
    if lo >= 0 and hi < 256:
        return "UTINYINT"
    if -32_768 <= lo and hi < 32_768:
        return "SMALLINT"
    if -(2**31) <= lo and hi < 2**31:
        return "INTEGER"
    return "BIGINT"

chosen = {c: pick_type(s) for c, s in stats.items()}

by_type = defaultdict(list)
for c, t in chosen.items():
    by_type[t].append(c)

for t, cols in sorted(by_type.items(), key=lambda x: -len(x[1])):
    print(f"{t:10s} {len(cols):>3d} {', '.join(cols[:6])}{' ...' if len(cols) > 6 else ''}")

UTINYINT   228 isFraud, card3, card5, addr2, C3, V1 ...
DOUBLE      79 TransactionAmt, D8, D9, V126, V127, V128 ...
SMALLINT    71 card1, card2, addr1, dist1, dist2, C1 ...
INTEGER      2 TransactionID, TransactionDT


In [6]:
# NULL 컬럼 - 제외 후보
by_type.get("ALL_NULL", [])

[]

In [7]:
# 절감량 확인
SIZE = {"UTINYINT": 1, "SMALLINT": 2, "INTEGER": 4, "BIGINT": 8, "DOUBLE": 8}
rows = con.execute("SELECT COUNT(*) FROM t").fetchone()[0]

naive = sum(8 for _ in numeric)
tuned = sum(SIZE.get(t, 8) for t in chosen.values())
print(f"행당 {naive:,} -> {tuned:,} 바이트 ({(1 - tuned / naive) * 100:.0f}% 절감)")
print(f"전체 {naive * rows / 1e9:.2f}GB -> {tuned * rows / 1e9:.2f}GB ({rows:,} 행)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

행당 3,040 -> 1,010 바이트 (67% 절감)
전체 3.34GB -> 1.11GB (1,097,231 행)


In [8]:
print("COLUMN_TYPES = {")
for c in inferred:
    t = chosen.get(c, inferred[c])
    if t != "ALL_NULL":
        print(f'    "{c}": "{t}",')
print("}")

COLUMN_TYPES = {
    "TransactionID": "INTEGER",
    "isFraud": "UTINYINT",
    "TransactionDT": "INTEGER",
    "TransactionAmt": "DOUBLE",
    "ProductCD": "VARCHAR",
    "card1": "SMALLINT",
    "card2": "SMALLINT",
    "card3": "UTINYINT",
    "card4": "VARCHAR",
    "card5": "UTINYINT",
    "card6": "VARCHAR",
    "addr1": "SMALLINT",
    "addr2": "UTINYINT",
    "dist1": "SMALLINT",
    "dist2": "SMALLINT",
    "P_emaildomain": "VARCHAR",
    "R_emaildomain": "VARCHAR",
    "C1": "SMALLINT",
    "C2": "SMALLINT",
    "C3": "UTINYINT",
    "C4": "SMALLINT",
    "C5": "SMALLINT",
    "C6": "SMALLINT",
    "C7": "SMALLINT",
    "C8": "SMALLINT",
    "C9": "SMALLINT",
    "C10": "SMALLINT",
    "C11": "SMALLINT",
    "C12": "SMALLINT",
    "C13": "SMALLINT",
    "C14": "SMALLINT",
    "D1": "SMALLINT",
    "D2": "SMALLINT",
    "D3": "SMALLINT",
    "D4": "SMALLINT",
    "D5": "SMALLINT",
    "D6": "SMALLINT",
    "D7": "SMALLINT",
    "D8": "DOUBLE",
    "D9": "DOUBLE",
    "D10": "S

In [9]:
# isFraud 가 test에서 NULL 로 유지되는지 확인
con.execute("""
    SELECT
        COUNT(*) AS total,
        COUNT(isFraud) AS labeled,
        COUNT(*) - COUNT(isFraud) AS unlabeled
    FROM t
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total,labeled,unlabeled
0,1097231,590540,506691


In [10]:
# M 컬럼 NULL 비율
con.execute("""
    SELECT
        COUNT(*) - COUNT(M1) AS m1_null,
        COUNT(*) - COUNT(M4) AS m4_null,
        COUNT(DISTINCT M4) AS m4_distinct
    FROM t
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,m1_null,m4_null,m4_distinct
0,447739,519189,3


## identity 컬럼

test_identity.csv 는 id-01 형태의 하이픈을 사용. 
정규화한 뒤에 타입을 판정해야함.

In [ ]:
from ieee_cis.etl.identity_fix import normalize_identity_columns

tr = f"{BASE}/train_identity.csv"
te = f"{BASE}/test_identity.csv"

# test 헤더를 정규화해서 alias 를 만듦
te_cols = [r[0] for r in con.execute(
    f"DESCRIBE SELECT * FROM read_csv('{te}', sample_size=-1)"
).fetchall()]
aliases = ", ".join(
    f'"{old}" AS "{new}"'
    for old, new in zip(te_cols, normalize_identity_columns(te_cols))
)

con.execute(f"""
    CREATE VIEW ident AS
    SELECT * FROM read_csv('{tr}', sample_size=-1)
    UNION ALL BY NAME
    SELECT {aliases} FROM read_csv('{te}', sample_size=-1)
""")

In [13]:
ident_inferred = {n: t for n, t, *_ in con.execute("DESCRIBE ident").fetchall()}
Counter(ident_inferred.values())

Counter({'DOUBLE': 23, 'VARCHAR': 13, 'BOOLEAN': 4, 'BIGINT': 1})

In [14]:
# transaction과 같은 방식으로 판정
ident_numeric = [c for c, t in ident_inferred.items() if t in ("DOUBLE", "BIGINT", "INTEGER")]
parts = [f'MIN("{c}"), MAX("{c}"), BOOL_AND("{c}" = FLOOR("{c}"))' for c in ident_numeric]
row = con.execute("SELECT " + ", ".join(parts) + " FROM ident").fetchone()

ident_chosen = dict(ident_inferred)
for i, c in enumerate(ident_numeric):
    ident_chosen[c] = pick_type(
        {"min": row[i * 3], "max": row[i * 3 + 1], "is_int": row[i * 3 + 2]}
    )

Counter(ident_chosen.values())

Counter({'SMALLINT': 14,
         'VARCHAR': 13,
         'UTINYINT': 7,
         'BOOLEAN': 4,
         'INTEGER': 2,
         'DOUBLE': 1})

In [15]:
# 커버리지 확인 - LEFT JOIN 후 has_identity 검증에 쓸 기준값
con.execute(f"""
    SELECT
        (SELECT COUNT(*) FROM ident) AS identity_rows,
        (SELECT COUNT(*) FROM t) AS txn_rows,
        (SELECT COUNT(*) FROM ident)::DOUBLE / (SELECT COUNT(*) FROM t) as coverage
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,identity_rows,txn_rows,coverage
0,286140,1097231,0.260784


In [16]:
print("IDENTITY_COLUMN_TYPES = {")
for c, t in ident_chosen.items():
    print(f'    "{c}": "{t}",')
print("}")

IDENTITY_COLUMN_TYPES = {
    "TransactionID": "INTEGER",
    "id_01": "SMALLINT",
    "id_02": "INTEGER",
    "id_03": "SMALLINT",
    "id_04": "SMALLINT",
    "id_05": "SMALLINT",
    "id_06": "SMALLINT",
    "id_07": "SMALLINT",
    "id_08": "SMALLINT",
    "id_09": "SMALLINT",
    "id_10": "SMALLINT",
    "id_11": "DOUBLE",
    "id_12": "VARCHAR",
    "id_13": "UTINYINT",
    "id_14": "SMALLINT",
    "id_15": "VARCHAR",
    "id_16": "VARCHAR",
    "id_17": "UTINYINT",
    "id_18": "UTINYINT",
    "id_19": "SMALLINT",
    "id_20": "SMALLINT",
    "id_21": "SMALLINT",
    "id_22": "UTINYINT",
    "id_23": "VARCHAR",
    "id_24": "UTINYINT",
    "id_25": "SMALLINT",
    "id_26": "UTINYINT",
    "id_27": "VARCHAR",
    "id_28": "VARCHAR",
    "id_29": "VARCHAR",
    "id_30": "VARCHAR",
    "id_31": "VARCHAR",
    "id_32": "UTINYINT",
    "id_33": "VARCHAR",
    "id_34": "VARCHAR",
    "id_35": "BOOLEAN",
    "id_36": "BOOLEAN",
    "id_37": "BOOLEAN",
    "id_38": "BOOLEAN",
    "Devic